# Project: Multi-agent wedding planner

**Module:** Module 2 — Advanced Agent Patterns  
**Notebook:** `notebooks/module-2/2.4_wedding_planners.ipynb`

## Purpose
Build a coordinator that delegates venue, playlist and related tasks to specialist agents using shared state and tools.

## Workflow
1. **Prepare the environment** — load required packages and local environment variables. Keep API keys in an ignored `.env` file; never paste credentials into notebook cells.
2. **Build the capability** — configure the model/agent, messages, tools, middleware or data source used in this lesson.
3. **Execute the example** — invoke or stream the agent and inspect the returned messages, tool calls, structured response or state.
4. **Validate behavior** — compare the result with the intended behavior, test an edge case, and check provider/service availability if execution fails.
5. **Reflect** — use the key learnings below to explain the design choices and limitations.

## Learning objectives
- Understand the concept demonstrated by this notebook and where it fits in an agent workflow.
- Explain the role of the main LangChain/LangGraph abstractions used in the implementation.
- Trace inputs through execution and interpret the resulting output.
- Identify operational concerns such as credentials, external services, state, safety and error handling.

## Key learnings
- MCP standardizes how an application exposes tools and resources to clients.
- Context is for runtime dependencies; state is for data that belongs to the agent execution/thread.
- Multi-agent design works best when each specialist has a narrow responsibility and clear input/output.

## Prerequisites and execution notes
- Use the repository setup instructions in the root `README.md` and the supported Python version.
- Configure only the provider credentials needed by this notebook in your local `.env` file.
- Some cells call external APIs, local MCP services, audio devices or sample databases; those dependencies must be available.
- Model output is non-deterministic and provider model names/API behavior can change. Treat outputs as examples, not fixed expected strings.

---



### Implementation step 1

Imports the libraries and APIs needed by the following example: from dotenv import load_dotenv. Loads local environment variables so provider/API credentials can be read without hard-coding secrets in the notebook.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from dotenv import load_dotenv. Loads local environment variables so provider/API credentials can be read without hard-coding secrets in the notebook.

from dotenv import load_dotenv

load_dotenv()


# 2.4 Wedding Planner
In this lab you will build a multi-agent wedding planner.

> Note: This lab has been updated since filming to make the agent performance more robust to errors and to limit run time. In particular: 1) added error handling for MCP failures, 2) limited the number of searches. It's worth noting that, where possible, tool errors should be returned to the agent rather than failing so that the agent can adjust its invocation and try again.

## Setup Tools


### Implementation step 4

Imports the libraries and APIs needed by the following example: import asyncio; from langchain_mcp_adapters.client import MultiServerMCPClient; from mcp.shared.exceptions import McpError (and 1 more import statement(s)) Defines a reusable Python function/tool. The docstring and type annotations communicate the tool purpose and input schema to the agent. Connects to an MCP server/client and discovers externally hosted tools. Verify server availability and transport configuration before running. Inspects or prints the result so you can verify the output structure and follow the execution path.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: import asyncio; from langchain_mcp_adapters.client import MultiServerMCPClient; from mcp.shared.exceptions import McpError (and 1 more import statement(s)) Defines a reusable Python function/tool. The docstring and type annotations communicate the tool purpose and input schema to the agent. Connects to an MCP server/client and discovers externally hosted tools. Verify server availability and transport configuration before running. Inspects or prints the result so you can verify the output structure and follow the execution path.

import asyncio

from langchain_mcp_adapters.client import MultiServerMCPClient
from mcp.shared.exceptions import McpError
from mcp.types import CallToolResult, TextContent

RETRYABLE_MCP_CODES = {-32603}

class RetryMCPInterceptor:
    """Intercept MCP tool calls: retry transient failures, surface all errors gracefully.

    - Retryable McpError codes (e.g. -32603): retry with exponential backoff.
    - Non-retryable McpError codes (e.g. -32602): return error message immediately.
    - Any other exception (fetch failed, network errors, etc.): retry then return error message.
    """

    def __init__(self, max_retries: int = 3):
        self.max_retries = max_retries

    async def __call__(self, request, handler):
        last_error = None
        for attempt in range(self.max_retries):
            try:
                return await handler(request)
            except McpError as exc:
                last_error = exc
                print(f"[MCP interceptor] {type(exc).__name__} on {request.name} "
                      f"(code {exc.error.code}, attempt {attempt+1}/{self.max_retries}): {exc}")
                if exc.error.code not in RETRYABLE_MCP_CODES:
                    return CallToolResult(
                        content=[TextContent(type="text", text=f"Tool call failed (non-retryable): {exc}")],
                        isError=False,
                    )
            except Exception as exc:
                last_error = exc
                print(f"[MCP interceptor] {type(exc).__name__} on {request.name} "
                      f"(attempt {attempt+1}/{self.max_retries}): {exc}")

            if attempt < self.max_retries - 1:
                await asyncio.sleep(2 ** attempt)

        print(f"[MCP interceptor] all {self.max_retries} retries exhausted for {request.name}")
        return CallToolResult(
            content=[TextContent(type="text", text=f"Tool call failed after {self.max_retries} attempts: {last_error}")],
            isError=False,
        )

client = MultiServerMCPClient(
    {
        "travel_server": {
                "transport": "streamable_http",
                "url": "https://mcp.kiwi.com"
            }
    },
    tool_interceptors=[RetryMCPInterceptor()],
)

tools = await client.get_tools()

### Implementation step 5

Imports the libraries and APIs needed by the following example: from typing import Dict, Any; from tavily import TavilyClient; from langchain.tools import tool. Defines a reusable Python function/tool. The docstring and type annotations communicate the tool purpose and input schema to the agent.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from typing import Dict, Any; from tavily import TavilyClient; from langchain.tools import tool. Defines a reusable Python function/tool. The docstring and type annotations communicate the tool purpose and input schema to the agent.

from typing import Dict, Any
from tavily import TavilyClient
from langchain.tools import tool

tavily_client = TavilyClient()

@tool
def web_search(query: str, search_number: int, max_search_number: int) -> Dict[str, Any]:
    """Search the web for information. You must track your search count by providing
    search_number (starting at 1) and max_search_number on every call.
    Queries must use only plain text characters. Do not use accented or special characters     
      (e.g., use 'capacite' instead of 'capacité').
    """
    if search_number > max_search_number:
        return {"message": "Search limit reached. Please summarize your findings and provide your final answer."}
    try:
        return tavily_client.search(query)
    except Exception as e:
        return {"error": str(e)}

### Implementation step 6

Imports the libraries and APIs needed by the following example: from langchain_community.utilities import SQLDatabase. Defines a reusable Python function/tool. The docstring and type annotations communicate the tool purpose and input schema to the agent. Connects to or queries a relational database. Validate/limit generated SQL and keep database credentials and sensitive data out of source control.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from langchain_community.utilities import SQLDatabase. Defines a reusable Python function/tool. The docstring and type annotations communicate the tool purpose and input schema to the agent. Connects to or queries a relational database. Validate/limit generated SQL and keep database credentials and sensitive data out of source control.

from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

@tool
def query_playlist_db(query: str) -> str:

    """Query the database for playlist information"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error querying database: {e}"

## Create State

### Implementation step 8

Imports the libraries and APIs needed by the following example: from langchain.agents import AgentState. Works with agent state, which carries structured information through the graph execution and may persist through a checkpoint.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from langchain.agents import AgentState. Works with agent state, which carries structured information through the graph execution and may persist through a checkpoint.

from langchain.agents import AgentState

class WeddingState(AgentState):
    origin: str
    destination: str
    guest_count: str
    genre: str

## Create Subagents


### Implementation step 10

Imports the libraries and APIs needed by the following example: from langchain.agents import create_agent. Configures a chat model or agent. Model choice, system instructions and attached tools determine the capabilities available in this example. Connects to an MCP server/client and discovers externally hosted tools. Verify server availability and transport configuration before running.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from langchain.agents import create_agent. Configures a chat model or agent. Model choice, system instructions and attached tools determine the capabilities available in this example. Connects to an MCP server/client and discovers externally hosted tools. Verify server availability and transport configuration before running.

from langchain.agents import create_agent

# Travel agent
travel_agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
    system_prompt="""
    You are a travel agent. Search for flights to the desired destination wedding location.
    You are not allowed to ask any more follow up questions, you must find the best flight options based on the following criteria:
    - Price (lowest, economy class)
    - Duration (shortest)
    - Date (time of year which you believe is best for a wedding at this location)
    To make things easy, only look for one ticket, one way.
    You may need to make multiple searches to iteratively find the best options.
    You will be given no extra information, only the origin and destination. It is your job to think critically about the best options.
    If the MCP tool fails, returns malformed output, or does not give you usable flight results, try the tool again.
    Once you have found the best options, let the user know your shortlist of options.
    """
)

### Implementation step 11

Configures a chat model or agent. Model choice, system instructions and attached tools determine the capabilities available in this example.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Configures a chat model or agent. Model choice, system instructions and attached tools determine the capabilities available in this example.

# Venue agent
venue_agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search],
    system_prompt="""
    You are a venue specialist. Search for venues in the desired location, and with the desired capacity.
    You are not allowed to ask any more follow up questions, you must find the best venue options based on the following criteria:
    - Price (lowest)
    - Capacity (exact match)
    - Reviews (highest)
    You may need to make multiple searches to iteratively find the best options. 
    You have a suggested limit of 12 web searches. Count every web_search call you make.
    After 12 searches, you should stop searching and summarize the best options you have
    found so far.
    """
)

### Implementation step 12

Configures a chat model or agent. Model choice, system instructions and attached tools determine the capabilities available in this example. Connects to or queries a relational database. Validate/limit generated SQL and keep database credentials and sensitive data out of source control.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Configures a chat model or agent. Model choice, system instructions and attached tools determine the capabilities available in this example. Connects to or queries a relational database. Validate/limit generated SQL and keep database credentials and sensitive data out of source control.

# Playlist agent
playlist_agent = create_agent(
    model="gpt-5-nano",
    tools=[query_playlist_db],
    system_prompt="""
    You are a playlist specialist. Query the sql database and curate the perfect playlist for a wedding given a genre.
    Once you have your playlist, calculate the total duration and cost of the playlist, each song has an associated price.
    If you run into errors when querying the database, try to fix them by making changes to the query.
    Do not come back empty handed, keep trying to query the db until you find a list of songs.

    This is a SQLite database. Before writing any data queries, first discover the schema.
    """
)

## Main Coordinator


### Implementation step 14

Imports the libraries and APIs needed by the following example: from langchain.tools import ToolRuntime; from langchain.messages import HumanMessage, ToolMessage; from langgraph.types import Command. Defines a reusable Python function/tool. The docstring and type annotations communicate the tool purpose and input schema to the agent. Executes the model/agent with an input payload. Inspect the returned object to understand messages, structured data, tool activity or state changes. Builds a typed conversation message. Message roles and ordering provide the context the model uses for its next response. Uses runtime context to access request-scoped dependencies or contextual values without treating them as model-authored input. Works with agent state, which carries structured information through the graph execution and may persist through a checkpoint. Demonstrates a pause/resume control path. Review the proposed action before resuming; approval should be explicit for consequential operations. Inspects or prints the result so you can verify the output structure and follow the execution path.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from langchain.tools import ToolRuntime; from langchain.messages import HumanMessage, ToolMessage; from langgraph.types import Command. Defines a reusable Python function/tool. The docstring and type annotations communicate the tool purpose and input schema to the agent. Executes the model/agent with an input payload. Inspect the returned object to understand messages, structured data, tool activity or state changes. Builds a typed conversation message. Message roles and ordering provide the context the model uses for its next response. Uses runtime context to access request-scoped dependencies or contextual values without treating them as model-authored input. Works with agent state, which carries structured information through the graph execution and may persist through a checkpoint. Demonstrates a pause/resume control path. Review the proposed action before resuming; approval should be explicit for consequential operations. Inspects or prints the result so you can verify the output structure and follow the execution path.

from langchain.tools import ToolRuntime
from langchain.messages import HumanMessage, ToolMessage
from langgraph.types import Command

@tool
async def search_flights(runtime: ToolRuntime) -> str:
    """Travel agent searches for flights to the desired destination wedding location."""
    origin = runtime.state["origin"]
    destination = runtime.state["destination"]
    response = await travel_agent.ainvoke({"messages": [HumanMessage(content=f"Find flights from {origin} to {destination}")]})
    return response['messages'][-1].content

@tool
def search_venues(runtime: ToolRuntime) -> str:
    """Venue agent chooses the best venue for the given location and capacity."""
    destination = runtime.state["destination"]
    capacity = runtime.state["guest_count"]
    query = f"Find wedding venues in {destination} for {capacity} guests"
    response = venue_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content

@tool
def suggest_playlist(runtime: ToolRuntime) -> str:
    """Playlist agent curates the perfect playlist for the given genre."""
    genre = runtime.state["genre"]
    query = f"Find {genre} tracks for wedding playlist"
    response = playlist_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content

@tool
def update_state(origin: str, destination: str, guest_count: str, genre: str, runtime: ToolRuntime) -> str:
    """Update the state when you know all of the values: origin, destination, guest_count, genre. 
    This tool must be called alone, without any other tool calls. It must complete and return to make,
    the information available to other tools."""
    return Command(update={
        "origin": origin, 
        "destination": destination, 
        "guest_count": guest_count, 
        "genre": genre, 
        "messages": [ToolMessage("Successfully updated state", tool_call_id=runtime.tool_call_id)]}
        )


### Implementation step 15

Imports the libraries and APIs needed by the following example: from langchain.agents import create_agent. Configures a chat model or agent. Model choice, system instructions and attached tools determine the capabilities available in this example. Works with agent state, which carries structured information through the graph execution and may persist through a checkpoint.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from langchain.agents import create_agent. Configures a chat model or agent. Model choice, system instructions and attached tools determine the capabilities available in this example. Works with agent state, which carries structured information through the graph execution and may persist through a checkpoint.

from langchain.agents import create_agent

coordinator = create_agent(
    model="gpt-5-nano",
    tools=[search_flights, search_venues, suggest_playlist, update_state],
    state_schema=WeddingState,
    system_prompt="""
    You are a wedding coordinator. 
    First find all the information you need to update the state. When you have the information, update the state.
    Once that has completed and returned, you can delegate the tasks 
    to your specialists for flights, venues, and playlists.
    Once you have received their answers, coordinate the perfect wedding for me.
    """
)


## Test


### Implementation step 17

Imports the libraries and APIs needed by the following example: from langchain.messages import HumanMessage. Executes the model/agent with an input payload. Inspect the returned object to understand messages, structured data, tool activity or state changes. Builds a typed conversation message. Message roles and ordering provide the context the model uses for its next response.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from langchain.messages import HumanMessage. Executes the model/agent with an input payload. Inspect the returned object to understand messages, structured data, tool activity or state changes. Builds a typed conversation message. Message roles and ordering provide the context the model uses for its next response.

from langchain.messages import HumanMessage

response = await coordinator.ainvoke(
    {
        "messages": [HumanMessage(content="I'm from London and I'd like a wedding in Paris for 100 guests, jazz-genre")],
    },
    config={"tags": ["WP"], "recursion_limit": 40},  #tag traces to make them easy to find in Langsmith. Increase number of steps the agent can take to 40.
)

### Implementation step 18

Imports the libraries and APIs needed by the following example: from pprint import pprint. Inspects or prints the result so you can verify the output structure and follow the execution path.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from pprint import pprint. Inspects or prints the result so you can verify the output structure and follow the execution path.

from pprint import pprint

pprint(response)

### Implementation step 19

Inspects or prints the result so you can verify the output structure and follow the execution path.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Inspects or prints the result so you can verify the output structure and follow the execution path.

print(response["messages"][-1].content)

link to trace: https://smith.langchain.com/public/7b5fe668-d3e3-4af4-b513-a8cacc0c9e84/r